In [25]:
import numpy as np

import pandas as pd
pd.set_option("display.max_columns", None)


In [2]:
dados = pd.read_csv('../data/klike_challenge_dataset.csv')
dados.head(5)

,campaign_id,date,platform,category,objective,target_audience_age,is_retargeting,video_duration_s,format,has_subtitle,has_cta,has_hook,has_face,text_density,music_voice_ratio,impressions,clicks,ctr,cpc,spend,conversions,revenue,roas,avg_watch_time_s,engagement_rate,klike_score
0,KLK-0001,2024-01-01,Meta,Branding,awareness,18-24,False,NaN,vertical,True,True,False,True,medium,0.39,150825,5882,0.0390,0.82,4823.24,48,0.00,0.00,3.9,0.0446,63.6
1,KLK-0002,2024-03-22,LinkedIn,App Install,traffic,25-34,False,NaN,horizontal,True,True,True,True,low,0.07,7433,493,0.0664,3.54,1745.22,34,NaN,0.16,6.4,0.0217,70.0
2,KLK-0003,2024-05-04,TikTok,SaaS,traffic,25-34,False,8.0,vertical,True,True,True,True,medium,0.43,193198,15069,0.0780,0.38,5726.22,492,51053.44,8.92,8.0,0.1000,99.1
3,KLK-0004,2024-12-27,TikTok,Lead Gen,engagement,25-34,False,27.0,quadrado,False,True,True,True,medium,0.54,6506,601,0.0925,0.71,426.71,19,1113.78,2.61,20.3,0.0414,74.9
4,KLK-0005,2024-07-10,Meta,Branding,engagement,25-34,False,34.0,vertical,True,True,True,True,low,0.35,63830,2355,0.0369,0.38,894.90,13,0.00,0.00,34.0,0.0184,77.9


In [3]:
df_hook = dados.groupby(['platform', 'has_hook'], observed=True)['roas'].agg(['count', 'median'])
com_hook = df_hook.xs(True, level='has_hook')['median']
sem_hook = df_hook.xs(False, level='has_hook')['median']

diferenca_pct = ((com_hook - sem_hook) / sem_hook) * 100
display(diferenca_pct.round(2))
df_hook

platform
LinkedIn     0.00
Meta       -27.07
TikTok      35.91
Name: median, dtype: float64

count  median
platform has_hook               
LinkedIn False        38    0.53
         True         45    0.53
Meta     False       113    1.81
         True        128    1.32
TikTok   False        93    2.20
         True         83    2.99

In [ ]:
# Funções ----------------------------------------------------------------------------------------------------------------------

def comparar_atributo(dados, atributo, metrica, atual, alternativa):
    df = dados.groupby(['platform', atributo], observed=True)[metrica].agg(['count', 'median'])
    grupo_atual = df.xs(atual, level=atributo)['median']
    grupo_alternativa = df.xs(alternativa, level=atributo)['median']
    diferenca_pct = ((grupo_alternativa - grupo_atual) / grupo_atual) * 100
    return diferenca_pct

def melhor_alternativa(dados, plataforma, atributo, metrica, valor_atual):
    df = dados[dados['platform'] == plataforma]
    resumo = df.groupby(atributo, observed=True)[metrica].agg(['count', 'median'])
    # Considera só grupos com pelo menos 10 campanhas
    resumo = resumo[resumo['count'] >= 10]

    if valor_atual not in resumo.index:
        return None

    mediana_atual = resumo.loc[valor_atual, 'median']
    # Remove a categoria atual das alternativas
    alternativas = resumo.drop(index=valor_atual)

    if alternativas.empty or mediana_atual <= 0:
        return None

    # Seleciona alternativa com maior mediana
    melhor_valor = alternativas['median'].idxmax()
    melhor_mediana = alternativas.loc[melhor_valor, 'median']

    diferenca_pct = (melhor_mediana - mediana_atual) / mediana_atual * 100

    if isinstance(valor_atual, bool) and isinstance(melhor_valor, bool):
        valor_atual = bool(valor_atual)
        melhor_valor = bool(melhor_valor)

    if diferenca_pct <= 0:
        return None

    return {'atributo': atributo, 'valor_atual': valor_atual, 'valor_recomendado': melhor_valor, 
            'metrica': metrica, 'diferenca_pct': float(diferenca_pct), 'mediana_atual': float(mediana_atual),
            'mediana_alternativa': float(melhor_mediana), 'n_atual': int(resumo.loc[valor_atual, 'count']),
            'n_alternativa': int(resumo.loc[melhor_valor, 'count'])}

def metrica_por_objetivo(objetivo):
    metricas_por_objetivo = {
    "traffic": "ctr",
    "engagement": "engagement_rate",
    "conversions": "roas",
    "awareness": "watch_time_ratio",
    "app_install": "conversion_rate"}
    return metricas_por_objetivo[objetivo]

def gerar_candidatas(campanha, dados):
    candidatas = []
    # Identificar a métrica principal da campanha
    metrica_principal = metrica_por_objetivo(campanha['objective'])
    metricas = ['ctr', 'conversion_rate', 'roas', 'watch_time_ratio', 'engagement_rate']
    metricas_ordenadas = [metrica_principal] + [m for m in metricas if m != metrica_principal]

    # Percorrer métricas e atributos modificáveis
    atributos = ["has_hook", "has_cta", "has_face", "has_subtitle", "text_density", "duration_group", "audio_group", "format"]
    for atributo in atributos:
        if pd.isna(campanha[atributo]):
            continue
        for metrica in metricas_ordenadas:
            resultado = melhor_alternativa(dados=dados, plataforma=campanha['platform'], atributo=atributo, metrica=metrica, 
                                                         valor_atual=campanha[atributo])
            if resultado is not None:
                resultado["metrica_principal"] = (metrica == metrica_principal)
                candidatas.append(resultado)
    
    return candidatas

# def priorizar_recomendacoes(candidatas):
    

In [ ]:
# Criando grupos de duração para tratar como categórica

dados['duration_group'] = pd.qcut(dados['video_duration_s'], q=3, labels=['short', 'medium', 'long'])

df_duration = dados.groupby(['platform', 'duration_group'], observed=True)['engagement_rate'].agg(['count', 'median'])
display(df_duration)


In [ ]:
# Criando grupos de audio para tratar como categórica

dados['audio_group'] = pd.qcut(dados['music_voice_ratio'], q=3, labels=['low_music', 'medium_music', 'high_music'])

df_audio = dados.groupby(['platform', 'audio_group'], observed=True)['engagement_rate'].agg(['count', 'median'])
display(df_audio)

dados['music_voice_ratio'].quantile([0, 1/3, 2/3, 1])

In [ ]:
# Criando taxa de conversão com base na quantidade de cliques

dados["conversion_rate"] = (dados["conversions"] / dados["clicks"].replace(0, np.nan))

display(dados["conversion_rate"].describe())
print("Valores ausentes:", dados["conversion_rate"].isna().sum())
print("Taxas acima de 100%:", (dados["conversion_rate"] > 1).sum())

In [ ]:
print(dados[["avg_watch_time_s", "video_duration_s"]].isna().sum())

print("Durações iguais a zero:",
      (dados["video_duration_s"] == 0).sum())

In [ ]:
# Criando a proporção do vídeo que foi assistida

dados["watch_time_ratio"] = dados["avg_watch_time_s"] / dados["video_duration_s"].replace(0, np.nan)

display(dados["watch_time_ratio"].describe())
print("Valores ausentes:", dados["watch_time_ratio"].isna().sum())
print("Proporções acima de 100%:", (dados["watch_time_ratio"] > 1).sum())

In [36]:
# CÉLULA DE TESTE --------------------------------------------------------------------------------------------
campanha = dados.iloc[4]

candidatas = gerar_candidatas(campanha, dados)

print(f"Total de candidatas: {len(candidatas)}")
display(pd.DataFrame(candidatas))

Total de candidatas: 24


,atributo,valor_atual,valor_recomendado,metrica,diferenca_pct,mediana_atual,mediana_alternativa,n_atual,n_alternativa,metrica_principal
0,has_hook,True,False,roas,37.121212,1.320000,1.810000,128,113,False
1,has_cta,True,False,ctr,4.746835,0.063200,0.066200,174,67,False
2,has_cta,True,False,conversion_rate,15.539320,0.027996,0.032346,174,67,False
3,has_cta,True,False,watch_time_ratio,9.559432,0.509617,0.558333,156,57,False
4,has_subtitle,True,False,engagement_rate,1.977401,0.017700,0.018050,119,86,True
5,has_subtitle,True,False,conversion_rate,10.590569,0.029204,0.032297,128,92,False
6,has_subtitle,True,False,roas,6.542056,1.605000,1.710000,128,92,False
7,has_subtitle,True,False,watch_time_ratio,4.504505,0.510345,0.533333,117,77,False
8,text_density,low,medium,engagement_rate,23.030303,0.016500,0.020300,97,96,True
9,text_density,low,medium,ctr,4.347826,0.067850,0.070800,104,105,False
